In [ ]:
from benchmark_pipeline.path import DATA
from benchmark_pipeline.config import (
    FEATURES,
    CITY_BBOXES,
    CITY_ECRS,
)
from benchmark_pipeline.generator.features import *

from benchmark_pipeline.loader.osm_loaders import load_tables
from benchmark_pipeline.loader.data_cleaning.name_matching import name_matching_cleaning
from benchmark_pipeline.generator.composite.generator_qcomposite import make_question_composite
from benchmark_pipeline.generator.composite.format_samples import format_composit_samples
from benchmark_pipeline.generator.register import register, BenchmarkConfig
from benchmark_pipeline.generator.multihop.generator_multihop import make_multihop_dataset

from benchmark_pipeline.generator.targets.mask_anchors import add_anchors_mask
from benchmark_pipeline.generator.targets.ellypses import add_ellypse
from benchmark_pipeline.generator.targets.anchors import add_anchors
from benchmark_pipeline.dataviz_tools.question_viz import montrer_questions

In [ ]:
import time

for city_name, bbox in CITY_BBOXES.items():
    city_start = time.perf_counter()
    print(f"\n=== [{list(CITY_BBOXES).index(city_name) + 1}/{len(CITY_BBOXES)}] {city_name} ===")
    conf = BenchmarkConfig(name=city_name, bbox=bbox, ecrs=CITY_ECRS[city_name], rows_target=[])

    print("1/5 chargement OSM...")
    step_start = time.perf_counter()
    df_osm, df_street, df_area = load_tables(conf.bbox, conf.ecrs)
    print(f"    {len(df_osm)} POIs, {len(df_street)} rues, {len(df_area)} zones ({time.perf_counter() - step_start:.0f} s)")

    print("2/5 dédoublonnage des noms...")
    df_all, (df_osm_clean, df_street_clean, df_area_clean) = name_matching_cleaning(df_osm, df_street, df_area)
    print(f"    gardés : {len(df_osm_clean)} POIs, {len(df_street_clean)} rues, {len(df_area_clean)} zones")

    print("3/5 génération des questions...")
    step_start = time.perf_counter()
    df_question = make_question_composite(
        df_osm_clean, 
        df_area_clean, 
        df_street_clean,
        nb_q_by_temp=conf.nb_q_by_temp, 
        nb_q_by_feat=conf.nb_q_by_feat,
        ratio_features=conf.ratio_features, 
        part_avec_features=conf.part_with_features,
        feature_cols=FEATURES, 
        seed=conf.seed, 
        is_paraphrase=conf.is_paraphrase
    )
    print(f"    {len(df_question)} questions ({time.perf_counter() - step_start:.0f} s)")

    print("4/5 enregistrement du benchmark...")
    step_start = time.perf_counter()
    register(
        DATA / conf.name, 
        df_question, 
        df_osm_clean, 
        df_street_clean, 
        df_area_clean, 
        df_all,
        conf, 
        format_composit_samples, 
        overwritte=True
    )
    print(f"    écrit dans {DATA / conf.name} ({time.perf_counter() - step_start:.0f} s)")

    print("5/5 questions multihop...")
    step_start = time.perf_counter()
    make_multihop_dataset(
        conf.nb_seed_questions, 
        conf.name, 
        conf.multihop_template,
        df_question, 
        df_osm_clean, 
        df_street_clean, 
        df_area_clean, 
        seed=conf.seed,
        nb_q_by_feat=conf.nb_q_by_feat, 
        ratio_features=conf.ratio_features,
        part_with_features=conf.part_with_features
    )
    print(f"    fait ({time.perf_counter() - step_start:.0f} s)")
    print(f"=== {city_name} terminé en {(time.perf_counter() - city_start) / 60:.1f} min ===")


In [ ]:
"""    #data= make_paraphrase(df_question_test, nb_generated=10)

    is_gseek_training = False
    if is_gseek_training:
        #sentence_model = SentenceEncoder()
        #df_question = add_anchors_mask(df_question, sentence_model)
        df_question = add_ellypse(df_question)
        df_question = add_anchors(df_question)
    #df_question = df_question.rename(columns={"pois": "results_poi_id"})
    # """

In [ ]:
#montrer_questions(df_question, df_osm, par_type=True)